In [1]:
import pandas as pd 
import sqlite3

conn = sqlite3.connect("../data/Chinook_Sqlite.sqlite")

tables = pd.read_sql_query(
"SELECT name FROM sqlite_master WHERE type='table';", conn
)
print(tables)

             name
0           Album
1          Artist
2        Customer
3        Employee
4           Genre
5         Invoice
6     InvoiceLine
7       MediaType
8        Playlist
9   PlaylistTrack
10          Track


### Explanation 

The first two lines load the tools.
sqlite3.connect opens the database,the same as in your test cell.
sqlite_master is a table SQLite keeps about itself listing everything in the database so this query asks "what tables exist?"

In [2]:
#look at the columns inside one table 

info = pd.read_sql_query(
"PRAGMA table_info(Customer);", conn
)
print(info)

    cid          name          type  notnull dflt_value  pk
0     0    CustomerId       INTEGER        1       None   1
1     1     FirstName  NVARCHAR(40)        1       None   0
2     2      LastName  NVARCHAR(20)        1       None   0
3     3       Company  NVARCHAR(80)        0       None   0
4     4       Address  NVARCHAR(70)        0       None   0
5     5          City  NVARCHAR(40)        0       None   0
6     6         State  NVARCHAR(40)        0       None   0
7     7       Country  NVARCHAR(40)        0       None   0
8     8    PostalCode  NVARCHAR(10)        0       None   0
9     9         Phone  NVARCHAR(24)        0       None   0
10   10           Fax  NVARCHAR(24)        0       None   0
11   11         Email  NVARCHAR(60)        1       None   0
12   12  SupportRepId       INTEGER        0       None   0


### Explanation 

PRAGMA table_info is a SQLite command that describes a table. You will get one row per column, with its *name*, its *type* (TEXT,INTEGER and so on), and *pk*, which is a 1 for the primary key. This is the same idea as the schemas.

In [3]:
#look at the columns in the tables
info =pd.read_sql_query("PRAGMA table_info(Invoice);", conn)
print(info)

   cid               name           type  notnull dflt_value  pk
0    0          InvoiceId        INTEGER        1       None   1
1    1         CustomerId        INTEGER        1       None   0
2    2        InvoiceDate       DATETIME        1       None   0
3    3     BillingAddress   NVARCHAR(70)        0       None   0
4    4        BillingCity   NVARCHAR(40)        0       None   0
5    5       BillingState   NVARCHAR(40)        0       None   0
6    6     BillingCountry   NVARCHAR(40)        0       None   0
7    7  BillingPostalCode   NVARCHAR(10)        0       None   0
8    8              Total  NUMERIC(10,2)        1       None   0


#### Foreign Key 

Customerid is the foreign key that links the table invoce and customer together.

In [4]:
#look at the columns in the tables
info =pd.read_sql_query("PRAGMA table_info(Track);", conn)
print(info)

   cid          name           type  notnull dflt_value  pk
0    0       TrackId        INTEGER        1       None   1
1    1          Name  NVARCHAR(200)        1       None   0
2    2       AlbumId        INTEGER        0       None   0
3    3   MediaTypeId        INTEGER        1       None   0
4    4       GenreId        INTEGER        0       None   0
5    5      Composer  NVARCHAR(220)        0       None   0
6    6  Milliseconds        INTEGER        1       None   0
7    7         Bytes        INTEGER        0       None   0
8    8     UnitPrice  NUMERIC(10,2)        1       None   0


In [5]:
#The actual data 
print(pd.read_sql_query("SELECT * FROM Customer LIMIT 5;", conn))

   CustomerId  FirstName     LastName  \
0           1       Luís    Gonçalves   
1           2     Leonie       Köhler   
2           3   François     Tremblay   
3           4      Bjørn       Hansen   
4           5  František  Wichterlová   

                                            Company  \
0  Embraer - Empresa Brasileira de Aeronáutica S.A.   
1                                               NaN   
2                                               NaN   
3                                               NaN   
4                                  JetBrains s.r.o.   

                           Address                 City State         Country  \
0  Av. Brigadeiro Faria Lima, 2170  São José dos Campos    SP          Brazil   
1          Theodor-Heuss-Straße 34            Stuttgart   NaN         Germany   
2                1498 rue Bélanger             Montréal    QC          Canada   
3                 Ullevålsveien 14                 Oslo   NaN          Norway   
4                

In [6]:
# First query 
#Way 1 : Cursor object
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM Track ;")
print(cursor.fetchall())

[(3503,)]


In [8]:
#way 2 : Pandas read_sql_query
print(pd.read_sql_query("SELECT COUNT(*) AS total_tracks FROM Track", conn))

   total_tracks
0          3503


In [10]:
# the first 10 rows of data
print(pd.read_sql_query("SELECT FirstName, LastName,Country FROM Customer LIMIT 10;",conn))

   FirstName     LastName         Country
0       Luís    Gonçalves          Brazil
1     Leonie       Köhler         Germany
2   François     Tremblay          Canada
3      Bjørn       Hansen          Norway
4  František  Wichterlová  Czech Republic
5     Helena         Holý  Czech Republic
6     Astrid       Gruber         Austria
7       Daan      Peeters         Belgium
8       Kara      Nielsen         Denmark
9    Eduardo      Martins          Brazil


In [11]:
# 5 counties with the most customers
print (pd.read_sql_query("SELECT Country, COUNT(*) AS total_customers FROM Customer GROUP BY Country ORDER BY total_customers DESC LIMIT 5;",conn))

   Country  total_customers
0      USA               13
1   Canada                8
2   France                5
3   Brazil                5
4  Germany                4


In [14]:
#A parameterized query
country = "Brazil"
df = pd.read_sql_query(
    "SELECT FirstName, LastName, City  FROM Customer WHERE Country = ?;", conn, params=(country,)
)
print(df)

   FirstName   LastName                 City
0       Luís  Gonçalves  São José dos Campos
1    Eduardo    Martins            São Paulo
2  Alexandre      Rocha            São Paulo
3    Roberto    Almeida       Rio de Janeiro
4   Fernanda      Ramos             Brasília


In [15]:
conn.close()